# Multi-Model AI Annotation Benchmark

Runs 3 LLMs on 10 placental GEO papers (the top disagreement papers) and compares results.

**Models:** Gemini 3.0 Flash Preview, GPT-5.4, Claude Sonnet 4.6  
**Output:** Per-question answer + confidence (high/medium/low) + reasoning (only when confidence is low)

In [ ]:
# Cell 1: Install dependencies
# Uncomment and run if needed:
# !pip install google-genai openai anthropic pandas openpyxl

In [33]:
# Cell 2: Configuration
import os

# ============================================================
# API KEYS — paste your keys between the quotes
# ============================================================
GEMINI_API_KEY = ""  # set locally before running
ANTHROPIC_API_KEY = ""  # set locally before running
OPENAI_API_KEY = ""  # set locally before running

# Set them in the environment (overwrite any stale values)
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# ============================================================
# The 10 disagreement papers
# ============================================================
TARGET_PAPERS = {
    "GSE234729": {"pmid": "37949031",  "pmcid": "PMC10843761"},
    "GSE98224":  {"pmid": "29507646",  "pmcid": "PMC5833042"},
    "GSE220877": {"pmid": "36744021",  "pmcid": "PMC9896899"},
    "GSE145357": {"pmid": "32640239",  "pmcid": "PMC7396155"},
    "GSE155750": {"pmid": "33784241",  "pmcid": "PMC8403268"},
    "GSE130339": {"pmid": "31658584",  "pmcid": "PMC6829352"},
    "GSE215421": {"pmid": "30611556",  "pmcid": "PMC7156023"},
    "GSE131729": {"pmid": "31917811",  "pmcid": "PMC6952106"},
    "GSE154829": {"pmid": "32721520",  "pmcid": "PMC7855285"},
    "GSE128381": {"pmid": "31110514",  "pmcid": "PMC6501552"},
}

# Model configurations
MODELS = {
    "gemini-3.0-flash-preview": {
        "provider": "google",
        "model_id": "gemini-3-flash-preview",
    },
    "gpt-5.4": {
        "provider": "openai",
        "model_id": "gpt-5.4",
    },
    "claude-sonnet-4.6": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-6",
    },
}

# Files
PROCESSED_PAPERS_JSON = "processed_papers.json"
OUTPUT_DIR = "benchmark_results"

# Verify which keys are ready
for name, env_key in [("Gemini", "GEMINI_API_KEY"), ("OpenAI", "OPENAI_API_KEY"), ("Anthropic", "ANTHROPIC_API_KEY")]:
    status = "READY" if os.environ.get(env_key) else "NOT SET"
    print(f"  {name:10s}: {status}")

print(f"\nTarget papers: {len(TARGET_PAPERS)}")
print(f"Models: {list(MODELS.keys())}")

  Gemini    : READY
  OpenAI    : READY
  Anthropic : READY

Target papers: 10
Models: ['gemini-3.0-flash-preview', 'gpt-5.4', 'claude-sonnet-4.6']


In [34]:
# Cell 3: Load the 10 papers from processed_papers.json
import json

with open(PROCESSED_PAPERS_JSON, "r", encoding="utf-8") as f:
    all_papers = json.load(f)

# Build lookup: PMCID -> full text
paper_lookup = {}
for p in all_papers:
    pmcid = p["pmcid"]
    text = " ".join(p.get("chunks", []))
    paper_lookup[pmcid] = text

# Extract our 10 papers
papers_to_annotate = {}
missing = []
for geo_id, info in TARGET_PAPERS.items():
    pmcid = info["pmcid"]
    if pmcid in paper_lookup:
        papers_to_annotate[geo_id] = {
            "pmcid": pmcid,
            "pmid": info["pmid"],
            "text": paper_lookup[pmcid],
            "char_count": len(paper_lookup[pmcid]),
        }
        print(f"  OK  {geo_id} ({pmcid}): {len(paper_lookup[pmcid]):,} chars")
    else:
        missing.append((geo_id, pmcid))
        print(f"  MISSING  {geo_id} ({pmcid})")

print(f"\nLoaded: {len(papers_to_annotate)}/10")
if missing:
    print(f"Missing: {missing}")

  OK  GSE234729 (PMC10843761): 2,164 chars
  OK  GSE98224 (PMC5833042): 43,163 chars
  OK  GSE220877 (PMC9896899): 38,963 chars
  OK  GSE145357 (PMC7396155): 85,617 chars
  OK  GSE155750 (PMC8403268): 1,874 chars
  OK  GSE130339 (PMC6829352): 72,322 chars
  OK  GSE215421 (PMC7156023): 6,201 chars
  OK  GSE131729 (PMC6952106): 56,687 chars
  OK  GSE154829 (PMC7855285): 1,617 chars
  OK  GSE128381 (PMC6501552): 36,974 chars

Loaded: 10/10


In [35]:
# Cell 4: The annotation prompt
#
# Same 24 questions as the original llm_parser.py, plus:
#   - confidence: high / medium / low per question
#   - reasoning: only included when confidence is "low"
#
# Uses string.Template ($paper_text) instead of .format() to avoid
# conflicts with JSON curly braces in the prompt.

from string import Template

EXTRACTION_PROMPT = Template('''\
You are an expert biomedical data extractor specializing in placental and pregnancy research.

Your task: read the paper text below and answer each question.

Return ONLY a single valid JSON object. No markdown fences, no commentary.

=== OUTPUT FORMAT ===

For EACH question, return an object with these fields:
  "answer":     your answer (see rules below)
  "confidence": "high", "medium", or "low"
  "reasoning":  a 1-2 sentence justification — ONLY include this field when confidence is "low"

Example structure:
{
  "Birthweight of offspring provided (yes/no)": {
    "answer": "Yes",
    "confidence": "high"
  },
  "Paternal Weight provided (yes/no)": {
    "answer": "No",
    "confidence": "low",
    "reasoning": "The paper mentions parental demographics but does not specifically list paternal weight in any table or methods section."
  }
}

=== ANSWER RULES ===

- Yes/No questions: return exactly "Yes" or "No".
  "Yes" means the paper EXPLICITLY provides or reports this data.
  "No" means it is not reported. If unclear or ambiguous, return "No".

- Pregnancy trimester: return one of "1st", "2nd", "3rd", "Term", "Premature",
  or an array if multiple are studied (e.g., ["1st", "3rd"]).
  "Term" = full-term delivery (~37-42 weeks).
  "Premature" = delivery before 37 weeks due to complications.
  If the paper only collects samples during pregnancy but delivery timing is not
  discussed, answer based on when samples were collected.

- GA fields (weeks): return a number (e.g., 38.5) or "Not Provided".
  If multiple values, return the mean or most representative value.

- List fields: return a JSON array of strings. Empty array [] if none.

- Free text fields (author name, email, hospital, country, topic):
  return the most specific value from the paper. "Not Provided" if absent.

- Do NOT invent data. Only report what is explicitly stated in the paper.

=== CONFIDENCE GUIDELINES ===

- "high":   the answer is clearly and explicitly stated in the paper
- "medium": the answer requires some inference but is well-supported
- "low":    the answer is ambiguous, requires significant interpretation,
             or the paper provides contradictory/incomplete information

=== QUESTIONS ===

1.  "Supervisor/Contact/Corresponding author name" — the designated corresponding author. If not explicit, use the last author.
2.  "Supervisor/Contact/Corresponding author email" — their email address.
3.  "Main topic of the publication" — a brief 1-2 sentence summary.
4.  "Pregnancy trimester (1st, 2nd, 3rd, term, premature)" — when samples were collected or delivery occurred.
5.  "Birthweight of offspring provided (yes/no)"
6.  "Gestational Age at delivery provided (yes/no)"
7.  "GA at delivery (weeks)" — numeric value or "Not Provided".
8.  "Gestational Age at sample collection provided (yes/no)"
9.  "GA at sample collection (weeks)" — numeric value or "Not Provided".
10. "Sex of Offspring Provided (yes/no)"
11. "Parity provided (yes/no)"
12. "Gravidity provided (yes/no)"
13. "Number of offspring per pregnancy provided (yes/no)"
14. "Self-reported race/ethnicity of mother provided (yes/no)"
15. "Genetic ancestry or genetic strain provided (yes/no)"
16. "Maternal Height provided (yes/no)"
17. "Maternal Pre-pregnancy Weight provided (yes/no)"
18. "Paternal Height provided (yes/no)"
19. "Paternal Weight provided (yes/no)"
20. "Maternal age at sample collection provided (yes/no)"
21. "Paternal age at sample collection provided (yes/no)"
22. "Samples from pregnancy complications collected" — "Yes" or "No".
23. "Mode of delivery provided (yes/no)"
24. "Pregnancy complications in data set (list)" — list of complications studied.
25. "Fetal complications listed (yes/no)"
26. "Fetal complications in data set (list)" — list of fetal complications.
27. "Other Phenotypes Provided (list)" — other key phenotypes studied.
28. "Hospital/Center where samples were collected"
29. "Country where samples were collected"

=== PAPER TEXT ===

$paper_text
''')

def build_prompt(paper_text: str) -> str:
    return EXTRACTION_PROMPT.substitute(paper_text=paper_text)

# Quick test
test = build_prompt("test paper text")
assert '"Birthweight' in test, "Prompt construction failed"
print(f"Prompt template ready ({len(test):,} chars with test text)")

Prompt template ready (3,987 chars with test text)


In [36]:
# Cell 5: Model API callers
import time
import re

MAX_RETRIES = 5
RETRY_DELAY = 2.0

def parse_json_response(text: str) -> dict:
    """Extract JSON from model response, handling markdown fences."""
    text = text.strip()
    # Strip markdown code fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def call_gemini(prompt: str, model_id: str) -> str:
    """Call Google Gemini API."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                ),
            )
            return resp.text
        except Exception as e:
            print(f"      Gemini attempt {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
    raise RuntimeError(f"Gemini failed after {MAX_RETRIES} retries")


def call_openai(prompt: str, model_id: str) -> str:
    """Call OpenAI API."""
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "You are an expert biomedical data extractor. Return only valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.0,
            )
            return resp.choices[0].message.content
        except Exception as e:
            print(f"      OpenAI attempt {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
    raise RuntimeError(f"OpenAI failed after {MAX_RETRIES} retries")


def call_anthropic(prompt: str, model_id: str) -> str:
    """Call Anthropic Claude API."""
    import anthropic

    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.messages.create(
                model=model_id,
                max_tokens=4096,
                messages=[
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
            )
            return resp.content[0].text
        except Exception as e:
            print(f"      Anthropic attempt {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
    raise RuntimeError(f"Anthropic failed after {MAX_RETRIES} retries")


# Dispatcher
CALLERS = {
    "google": call_gemini,
    "openai": call_openai,
    "anthropic": call_anthropic,
}

print("Model callers ready.")

Model callers ready.


In [41]:
# Cell 6: Run all models on all papers
import pathlib

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Results structure: {model_name: {geo_id: parsed_json}}
all_results = {}

for model_name, model_cfg in MODELS.items():
    provider = model_cfg["provider"]
    model_id = model_cfg["model_id"]
    caller = CALLERS[provider]

    # Check if API key is set
    key_name = {"google": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[provider]
    if not os.environ.get(key_name):
        print(f"\n*** SKIPPING {model_name} — {key_name} not set ***")
        continue

    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({model_id})")
    print(f"{'='*60}")

    model_results = {}
    model_dir = os.path.join(OUTPUT_DIR, model_name)
    pathlib.Path(model_dir).mkdir(parents=True, exist_ok=True)

    for i, (geo_id, paper_info) in enumerate(papers_to_annotate.items(), 1):
        pmcid = paper_info["pmcid"]
        out_file = os.path.join(model_dir, f"{geo_id}.json")

        # Skip if already done (resume support)
        if os.path.exists(out_file):
            print(f"  [{i}/10] {geo_id} — already done, loading from disk")
            with open(out_file) as f:
                model_results[geo_id] = json.load(f)
            continue

        print(f"  [{i}/10] {geo_id} ({pmcid}, {paper_info['char_count']:,} chars)...")

        prompt = build_prompt(paper_info["text"])

        try:
            raw_response = caller(prompt, model_id)
            parsed = parse_json_response(raw_response)
            model_results[geo_id] = parsed

            # Save raw + parsed
            with open(os.path.join(model_dir, f"{geo_id}.raw.txt"), "w") as f:
                f.write(raw_response)
            with open(out_file, "w") as f:
                json.dump(parsed, f, indent=2)

            print(f"    OK — {len(parsed)} fields extracted")

        except Exception as e:
            print(f"    FAILED: {e}")
            # Save error
            with open(os.path.join(model_dir, f"{geo_id}.error.txt"), "w") as f:
                f.write(str(e))

        # Polite delay between API calls
        time.sleep(1.0)

    all_results[model_name] = model_results
    print(f"\n{model_name}: {len(model_results)}/10 papers completed")

print(f"\n\nAll done. Results saved to {OUTPUT_DIR}/")


Running: gemini-3.0-flash-preview (gemini-3-flash-preview)
  [1/10] GSE234729 — already done, loading from disk
  [2/10] GSE98224 — already done, loading from disk
  [3/10] GSE220877 — already done, loading from disk
  [4/10] GSE145357 — already done, loading from disk
  [5/10] GSE155750 — already done, loading from disk
  [6/10] GSE130339 — already done, loading from disk
  [7/10] GSE215421 — already done, loading from disk
  [8/10] GSE131729 — already done, loading from disk
  [9/10] GSE154829 — already done, loading from disk
  [10/10] GSE128381 — already done, loading from disk

gemini-3.0-flash-preview: 10/10 papers completed

Running: gpt-5.4 (gpt-5.4)
  [1/10] GSE234729 — already done, loading from disk
  [2/10] GSE98224 — already done, loading from disk
  [3/10] GSE220877 — already done, loading from disk
  [4/10] GSE145357 — already done, loading from disk
  [5/10] GSE155750 — already done, loading from disk
  [6/10] GSE130339 — already done, loading from disk
  [7/10] GSE215

In [ ]:
# Cell 6b: Normalize all model outputs into canonical format
import re

# ── Question categories ──
YESNO_QUESTIONS = [
    "Birthweight of offspring provided (yes/no)",
    "Gestational Age at delivery provided (yes/no)",
    "Gestational Age at sample collection provided (yes/no)",
    "Sex of Offspring Provided (yes/no)",
    "Parity provided (yes/no)",
    "Gravidity provided (yes/no)",
    "Number of offspring per pregnancy provided (yes/no)",
    "Self-reported race/ethnicity of mother provided (yes/no)",
    "Genetic ancestry or genetic strain provided (yes/no)",
    "Maternal Height provided (yes/no)",
    "Maternal Pre-pregnancy Weight provided (yes/no)",
    "Paternal Height provided (yes/no)",
    "Paternal Weight provided (yes/no)",
    "Maternal age at sample collection provided (yes/no)",
    "Paternal age at sample collection provided (yes/no)",
    "Samples from pregnancy complications collected",
    "Mode of delivery provided (yes/no)",
    "Fetal complications listed (yes/no)",
]

GA_QUESTIONS = [
    "GA at delivery (weeks)",
    "GA at sample collection (weeks)",
]

LIST_QUESTIONS = [
    "Pregnancy complications in data set (list)",
    "Fetal complications in data set (list)",
    "Other Phenotypes Provided (list)",
]

TRIMESTER_QUESTION = "Pregnancy trimester (1st, 2nd, 3rd, term, premature)"

TRIMESTER_MAP = {
    "1st": {"1", "1st", "first", "first trimester"},
    "2nd": {"2", "2nd", "second", "second trimester"},
    "3rd": {"3", "3rd", "third", "third trimester"},
    "Term": {"term", "full-term", "full term", "at term"},
    "Premature": {"premature", "preterm", "pre-term", "early", "ptb"},
}


def norm_yesno(val):
    s = str(val).strip().lower()
    if any(w in s for w in ["yes", "true", "provided", "reported", "available"]):
        return "Yes"
    return "No"


def norm_trimester_single(s):
    s = s.strip().lower()
    for canon, variants in TRIMESTER_MAP.items():
        if s in variants:
            return canon
    if "1" in s or "first" in s: return "1st"
    if "2" in s or "second" in s: return "2nd"
    if "3" in s or "third" in s: return "3rd"
    if "term" in s and "pre" not in s: return "Term"
    if "pre" in s or "premature" in s: return "Premature"
    return s.title()


def norm_trimester(val):
    if isinstance(val, list):
        return sorted(set(norm_trimester_single(str(v)) for v in val))
    s = str(val).strip()
    if not s or s.lower() in {"no", "nan", "none", "not provided", "n/a"}:
        return "Not Provided"
    if "," in s or "[" in s:
        items = re.findall(r"[a-zA-Z0-9]+(?:st|nd|rd|th)?", s.lower())
        if items:
            return sorted(set(norm_trimester_single(i) for i in items))
    return norm_trimester_single(s)


def norm_ga(val):
    if isinstance(val, (int, float)):
        return round(float(val), 1)
    s = str(val).strip().lower()
    if s in {"no", "nan", "none", "not provided", "n/a", ""}:
        return "Not Provided"
    m = re.search(r"(\d+\.?\d*)", s)
    if m:
        return round(float(m.group(1)), 1)
    return "Not Provided"


def norm_list(val):
    if isinstance(val, list):
        items = [str(v).strip().lower() for v in val if str(v).strip()]
        return sorted(set(items))
    s = str(val).strip()
    if s.lower() in {"no", "nan", "none", "not provided", "n/a", "[]", ""}:
        return []
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return sorted(set(str(v).strip().lower() for v in parsed if str(v).strip()))
    except (json.JSONDecodeError, ValueError):
        pass
    items = re.split(r"[;,]", s)
    return sorted(set(i.strip().lower() for i in items if i.strip()))


def norm_confidence(val):
    s = str(val).strip().lower()
    if s in {"high", "medium", "low"}: return s
    if "high" in s: return "high"
    if "med" in s: return "medium"
    if "low" in s: return "low"
    return "medium"


def norm_freetext(val):
    s = str(val).strip()
    if s.lower() in {"no", "nan", "none", "not provided", "n/a", "unknown", ""}:
        return "Not Provided"
    return s


def normalize_result(raw_result: dict) -> dict:
    normalized = {}
    for question, val in raw_result.items():
        if isinstance(val, dict):
            raw_answer = val.get("answer", "")
            raw_conf = val.get("confidence", "medium")
            raw_reason = val.get("reasoning", "")
        else:
            raw_answer = val
            raw_conf = "medium"
            raw_reason = ""

        if question in YESNO_QUESTIONS:
            answer = norm_yesno(raw_answer)
        elif question == TRIMESTER_QUESTION:
            answer = norm_trimester(raw_answer)
        elif question in GA_QUESTIONS:
            answer = norm_ga(raw_answer)
        elif question in LIST_QUESTIONS:
            answer = norm_list(raw_answer)
        else:
            answer = norm_freetext(raw_answer)

        entry = {"answer": answer, "confidence": norm_confidence(raw_conf)}
        if raw_reason and str(raw_reason).strip():
            entry["reasoning"] = str(raw_reason).strip()
        normalized[question] = entry

    return normalized


# ── Apply normalization ──
all_results_normalized = {}
for model_name, model_results in all_results.items():
    normalized_model = {}
    for geo_id, raw_result in model_results.items():
        normalized_model[geo_id] = normalize_result(raw_result)
    all_results_normalized[model_name] = normalized_model
    print(f"Normalized {model_name}: {len(normalized_model)} papers")

print(f"\nDone. Use 'all_results_normalized' for comparisons.")

In [ ]:
# Cell 8: Build comparison DataFrame
import pandas as pd

model_names = list(all_results_normalized.keys())
print(f"Models: {model_names}")

# All 29 questions
ALL_QUESTIONS = [
    "Supervisor/Contact/Corresponding author name",
    "Supervisor/Contact/Corresponding author email",
    "Main topic of the publication",
    "Pregnancy trimester (1st, 2nd, 3rd, term, premature)",
    "Birthweight of offspring provided (yes/no)",
    "Gestational Age at delivery provided (yes/no)",
    "GA at delivery (weeks)",
    "Gestational Age at sample collection provided (yes/no)",
    "GA at sample collection (weeks)",
    "Sex of Offspring Provided (yes/no)",
    "Parity provided (yes/no)",
    "Gravidity provided (yes/no)",
    "Number of offspring per pregnancy provided (yes/no)",
    "Self-reported race/ethnicity of mother provided (yes/no)",
    "Genetic ancestry or genetic strain provided (yes/no)",
    "Maternal Height provided (yes/no)",
    "Maternal Pre-pregnancy Weight provided (yes/no)",
    "Paternal Height provided (yes/no)",
    "Paternal Weight provided (yes/no)",
    "Maternal age at sample collection provided (yes/no)",
    "Paternal age at sample collection provided (yes/no)",
    "Samples from pregnancy complications collected",
    "Mode of delivery provided (yes/no)",
    "Pregnancy complications in data set (list)",
    "Fetal complications listed (yes/no)",
    "Fetal complications in data set (list)",
    "Other Phenotypes Provided (list)",
    "Hospital/Center where samples were collected",
    "Country where samples were collected",
]

rows = []
for geo_id in TARGET_PAPERS:
    for question in ALL_QUESTIONS:
        row = {"GEO_ID": geo_id, "Question": question}
        answers = []
        for model_name in model_names:
            result = all_results_normalized.get(model_name, {}).get(geo_id, {}).get(question, {})
            ans = result.get("answer", "N/A") if isinstance(result, dict) else "N/A"
            conf = result.get("confidence", "N/A") if isinstance(result, dict) else "N/A"
            reason = result.get("reasoning", "") if isinstance(result, dict) else ""
            row[f"{model_name}__answer"] = str(ans)
            row[f"{model_name}__confidence"] = str(conf)
            row[f"{model_name}__reasoning"] = str(reason)
            answers.append(str(ans))
        # Check if all models agree
        row["all_agree"] = "Yes" if len(set(answers)) == 1 else "No"
        rows.append(row)

comparison_df = pd.DataFrame(rows)
print(f"\nComparison table: {comparison_df.shape[0]} rows x {comparison_df.shape[1]} columns")
print(f"Questions per paper: {len(ALL_QUESTIONS)}")
print(f"Papers: {len(TARGET_PAPERS)}")

# Quick agreement summary
n_agree = (comparison_df["all_agree"] == "Yes").sum()
n_total = len(comparison_df)
print(f"\nOverall agreement: {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%)")

In [ ]:
# Cell 9: Summary statistics

if len(model_names) >= 2:
    print("=" * 60)
    print("YES/NO QUESTION AGREEMENT ACROSS MODELS")
    print("=" * 60)

    yesno_df = comparison_df[comparison_df["Question"].isin(YESNO_QUESTIONS)].copy()

    # Overall agreement rate
    n_total = len(yesno_df)
    n_agree = (yesno_df["all_agree"] == "Yes").sum()
    print(f"\nOverall: {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%) questions all models agree\n")

    # Per-question agreement
    print("Per question:")
    q_agree = yesno_df.groupby("Question")["all_agree"].apply(lambda x: (x=="Yes").mean())
    q_agree = q_agree.sort_values(ascending=True)
    for q, rate in q_agree.items():
        short_q = q[:60]
        bar = "*" * int(rate * 20)
        print(f"  {rate:5.0%}  {bar:<20s}  {short_q}")

    # Per-model pairwise agreement
    print("\nPairwise agreement (yes/no questions):")
    for i, m1 in enumerate(model_names):
        for m2 in model_names[i+1:]:
            a1 = yesno_df[f"{m1}__answer"].apply(norm_yesno)
            a2 = yesno_df[f"{m2}__answer"].apply(norm_yesno)
            agree = (a1 == a2).mean()
            print(f"  {m1} vs {m2}: {agree:.1%}")

    # Confidence distribution per model
    print("\nConfidence distribution:")
    for model_name in model_names:
        conf_col = f"{model_name}__confidence"
        if conf_col in comparison_df.columns:
            counts = comparison_df[conf_col].str.lower().value_counts()
            total = counts.sum()
            print(f"  {model_name}:")
            for level in ["high", "medium", "low"]:
                n = counts.get(level, 0)
                print(f"    {level}: {n}/{total} ({100*n/total:.0f}%)")

else:
    print("Need at least 2 models to compare. Run more models in Cell 6.")

In [39]:
# Cell 9: Save results to Excel

output_file = os.path.join(OUTPUT_DIR, "model_comparison.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # Sheet 1: Full comparison (all questions, all models)
    comparison_df.to_excel(writer, sheet_name="Full Comparison", index=False)

    # Sheet 2: Yes/No only (easier to scan)
    yesno_only = comparison_df[comparison_df["Question"].isin(YESNO_QUESTIONS)].copy()
    yesno_only.to_excel(writer, sheet_name="YesNo Questions", index=False)

    # Sheet 3: Disagreements only
    if "all_agree" in comparison_df.columns:
        disagree = comparison_df[comparison_df.get("all_agree") == "No"].copy()
        disagree.to_excel(writer, sheet_name="Disagreements", index=False)

    # Sheet 4: Low confidence answers (where models are uncertain)
    low_conf_rows = []
    for _, row in comparison_df.iterrows():
        for model_name in model_names:
            conf = str(row.get(f"{model_name}__confidence", "")).lower()
            if conf == "low":
                low_conf_rows.append({
                    "GEO_ID": row["GEO_ID"],
                    "Question": row["Question"],
                    "Model": model_name,
                    "Answer": row[f"{model_name}__answer"],
                    "Reasoning": row.get(f"{model_name}__reasoning", ""),
                })
    if low_conf_rows:
        low_df = pd.DataFrame(low_conf_rows)
        low_df.to_excel(writer, sheet_name="Low Confidence", index=False)

    # Sheet 5: Per-model raw answers (pivot: GEO_ID x Question for each model)
    for model_name in model_names:
        ans_col = f"{model_name}__answer"
        if ans_col in comparison_df.columns:
            pivot = comparison_df.pivot(
                index="GEO_ID", columns="Question", values=ans_col
            )
            sheet_name = model_name[:31]  # Excel 31-char sheet name limit
            pivot.to_excel(writer, sheet_name=sheet_name)

print(f"Results saved to: {output_file}")

IndexError: At least one sheet must be visible

In [ ]:
# Cell 11: Quick visual — disagreement heatmap
import matplotlib.pyplot as plt
import numpy as np

if len(model_names) >= 2 and "all_agree" in comparison_df.columns:
    yesno_df = comparison_df[comparison_df["Question"].isin(YESNO_QUESTIONS)].copy()

    # Build a matrix: rows = questions, cols = GEO IDs, value = agree(1) / disagree(0)
    pivot = yesno_df.pivot(index="Question", columns="GEO_ID", values="all_agree")
    matrix = pivot.map(lambda x: 1 if x == "Yes" else 0)

    # Shorten question labels
    short_labels = [q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")[:45] for q in pivot.index]

    fig, ax = plt.subplots(figsize=(14, 10))
    cmap = plt.cm.colors.ListedColormap(["#e74c3c", "#2ecc71"])
    im = ax.imshow(matrix.values, cmap=cmap, aspect="auto", vmin=0, vmax=1)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(short_labels)))
    ax.set_yticklabels(short_labels, fontsize=9)

    ax.set_title("Model Agreement on Yes/No Questions\n(Green = all agree, Red = disagreement)", fontsize=14)

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "agreement_heatmap.png")
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print(f"Saved: {fig_path}")
else:
    print("Need at least 2 models with results to plot.")